In [1]:
import pandas as pd
from pathlib import Path

import json
index_path = Path(r"C:\Users\craig\Documents\CS 125\Juke_Jam\indexes\indexes.json")
profiles_path = Path(r"C:\Users\craig\Documents\CS 125\Juke_Jam\indexes\time_context_profiles.json")


with open(index_path) as f:
    index_data = json.load(f)

genre_index = index_data["genre"]
mood_index = index_data["mood"]
energy_index = index_data["energy"]
artist_index = index_data["artist"]
title_index = index_data["title"]

with open(profiles_path) as f:
    time_context_profiles = json.load(f)

In [41]:
# Feature Vectors

# Feature set
features = set()

# genre features
for g in genre_index:
    features.add(f"genre:{g}")

# mood features
for m in mood_index:
    features.add(f"mood:{m}")

# energy features
for e in energy_index:
    features.add(f"energy:{e}")

# artist features
for a in artist_index:
    features.add(f"artist:{a}")

# title features
for t in title_index:
    features.add(f"title:{t}")

features = sorted(features)

feature_inds = {f: i for i, f in enumerate(features)}


In [3]:
# Song feature vectors
import numpy as np
from collections import defaultdict

num_features = len(features)
song_vectors = defaultdict(dict)

def update_song_features(song_id, feature):
    song_id = str(song_id)

    if feature not in feature_inds:
        return
        
    song_vectors[song_id][feature_inds[feature]] = 1

In [4]:
# Create song feature vectors
def populate_index(index, prefix=None):
    for feature, songs in index.items():
        f = f"{prefix}:{feature}" if prefix else feature

        if f not in feature_inds:
            print("Missing feature in vocab:", f)
            raise ValueError("Feature mismatch")

        for s in songs:
            update_song_features(s, f)


populate_index(genre_index, "genre")
populate_index(mood_index, "mood")
populate_index(energy_index, "energy")
populate_index(artist_index, "artist")
populate_index(title_index, "title")

In [40]:
# Compute IDF Weights
song_vector_num = len(song_vectors)
idf = {}

df_counts = defaultdict(int)

for vec in song_vectors.values():
    for key in vec.keys():
        df_counts[key] += 1

for f, i in feature_inds.items():
    df = df_counts.get(i, 0)
    idf[f] = np.log(song_vector_num / (1 + df))



In [39]:

for song, vec in song_vectors.items():
    for i in list(vec.keys()):
        feature = features[i]
        vec[i] *= idf[feature]

In [53]:
def build_query_vector(genres=None, mood=None, energy=None, 
                       artist=None, title=None, 
                       user_id=None, time_of_day=None, 
                       user_weight=1.0, time_weight=0.3, 
                       user_time_profiles=None):
    q = {}

    def set_feature(f, w):
        if f in feature_inds:
            q[feature_inds[f]] = q.get(i, 0) + w
    
    # User Query
    if genres:
        for g in genres:
            set_feature(f"genre:{g}", user_weight)
    
    if mood:
        set_feature(f"mood:{mood}", user_weight)
    
    if energy:
        set_feature(f"energy:{energy}", user_weight)

    if artist:
        set_feature(f"artist:{artist}", user_weight)
    
    if title:
        for token in title.split():
            set_feature(f"title:{token.lower()}")
    
    # Time Context (Weighted)
    profile = None
    if user_id and user_time_profiles:
        profile = user_time_profiles.get(user_id, {}).get(time_of_day, None)
    
    if profile:
        # Genres
        top_genres = profile.get("top_genres", [])
        for g in top_genres:
            set_feature(f"genre:{g}", time_weight)

        # Mood
        typical_mood = profile.get("typical_mood", [])
        if typical_mood:
            set_feature(f"mood:{typical_mood}", time_weight)

        # Activity or Energy
        # typical_activity = profile.get("typical_activity", [])
        # if typical_activity:
        #     set_feature(f"activity:{typical_activity}", time_weight)
    
    return q

In [8]:
def retrieve_candidates(genres=None, mood=None, energy=None,
                        artist=None, title=None):

    sets = []

    def add_hits(index, keys):
        if keys:
            hits = set()
            for k in keys:
                hits |= set(index.get(k, []))
            sets.append(hits)

    add_hits(genre_index, genres)
    add_hits(mood_index, [mood] if mood else None)
    add_hits(energy_index, [energy] if energy else None)
    add_hits(artist_index, [artist] if artist else None)

    if title:
        tokens = title.split()
        add_hits(title_index, tokens)

    if not sets:
        return []

    return list(set.union(*sets))


In [9]:
# Cosine Similarity
# cos(x) = AB/(||A|||B||)
# def cosine_similarity(a, b):
#     n_a = np.linalg.norm(a) == 0
#     n_b = np.linalg.norm(b) == 0
#     if n_a == 0 or n_b == 0:
#         return 0
#     return np.dot(a, b) / (n_a * n_b)

# Cosine sim for sparse vector
def cosine_similarity(q_vec, song_vec):
    dot = 0

    for i, val in song_vec.items():
        if i in q_vec:
            dot += val*q_vec[i]
    
    norm_q = np.sqrt(sum(v*v for v in q_vec.values()))
    norm_song = np.sqrt(sum(v*v for v in song_vec.values()))

    if norm_q == 0 or norm_song == 0:
        return 0
    
    return dot / (norm_q * norm_song)

In [49]:
# Ranking
def rank_songs(query, top_k=10, user_id=None, 
               time_of_day=None,
               user_weight=1.0, time_weight=0.3):
    
    candidates = retrieve_candidates(**query)

    if not candidates:
        return []
    
    query_vec = build_query_vector(**query,
                                   user_id=user_id,
                                   time_of_day=time_of_day,
                                   user_weight=user_weight,
                                   time_weight=time_weight,
                                   user_time_profiles=time_context_profiles)

    scored = []
    for s in candidates:
        score = cosine_similarity(query_vec, song_vectors[s])
        scored.append((s, score))

    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_k]


In [27]:
# Testing
# Song Info Lookup

import csv
song_metadata = {}

catalog_path = Path(r"C:\Users\craig\Documents\CS 125\Juke_Jam\data\processed\SONG_CATALOG.csv")

def fetch_song_metadata(song_ids, path=catalog_path):
    result = {}
    song_ids_set = set(song_ids)
    
    with open(path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            song_id = str(row["track_id"])
            if song_id in song_ids_set:
                result[song_id] = row
                if len(result) == len(song_ids_set):
                    break  # Stop once all requested IDs are found
    return result


In [42]:
# Print function
def print_result_data(results):
    song_ids = [song_id for song_id, _ in results]
    song_data = fetch_song_metadata(song_ids)

    result_data = []
    for song_id, score in results:
        data = song_data.get(song_id, {})
        result_data.append({
            "song_id": song_id,
            "score": score,
            "title": data.get("title", "Unknown"),
            "artist": data.get("artist_name", "Unknown"),
            "genre": data.get("genre", "Unknown"),
            "mood": data.get("mood_bucket", "Unknown"),
            "energy": data.get("energy", "Unknown")
        })
    
    for i, r in enumerate(result_data, 1):
        print(f"{i}. {r['title']} - {r['artist']} (ID: {r['song_id']})")
        print(f"   Score: {r['score']}")
        print(f"   Genre: {r['genre']}")
        print(f"   Mood: {r['mood']}")
        print(f"   Energy: {r['energy']}")


In [55]:
# Ranking Test 1: One parameter, no context
query = {
    "genres": ["acoustic"],
    "mood": "",
    "energy": ""
}

results = rank_songs(query, top_k=5)

print_result_data(results)

1. Love - Matt White (ID: 3mDvi0k4LuCA7ViLf3Qb3O)
   Score: 0.3596091385199028
   Genre: acoustic
   Mood: happy
   Energy: 0.684
2. Heaven - Acoustic - Grace George (ID: 3Re0unscFRNOkwwEGtxhCO)
   Score: 0.24848645243762052
   Genre: acoustic
   Mood: chill
   Energy: 0.24
3. Little Life - Frank Turner (ID: 5WhHTK5wyuY14tiHWuoH4G)
   Score: 0.2465277784069741
   Genre: acoustic
   Mood: focus
   Energy: 0.693
4. Beautiful In White - Matt Johnson (ID: 2K3jLUSLeHrPMA3nKvP1jT)
   Score: 0.24450868926865066
   Genre: acoustic
   Mood: sad
   Energy: 0.278
5. Believe - Acoustic - John Adams (ID: 4Gcx0DkgBVticyPDpezwg5)
   Score: 0.2421401983206199
   Genre: acoustic
   Mood: sad
   Energy: 0.241


In [58]:
# Ranking Test 2: Multiple features
query = {
    "genres": ["acoustic", "electronic"],
    "mood": "sad",
    "energy": "low"
}

results = rank_songs(query, top_k=5)

print_result_data(results)

1. Love - Matt White (ID: 3mDvi0k4LuCA7ViLf3Qb3O)
   Score: 0.20762043292751528
   Genre: acoustic
   Mood: happy
   Energy: 0.684
2. No - Miranda! (ID: 2lo0aKrsrO5lB4eFJ2yG3q)
   Score: 0.20210517869961717
   Genre: electronic
   Mood: focus
   Energy: 0.564
3. Quiero - Miranda! (ID: 23MbmgpUSb10XQjJv2zZEt)
   Score: 0.16724090832420507
   Genre: electronic
   Mood: happy
   Energy: 0.755
4. Beautiful In White - Matt Johnson (ID: 2K3jLUSLeHrPMA3nKvP1jT)
   Score: 0.16070401187968397
   Genre: acoustic
   Mood: sad
   Energy: 0.278
5. Believe - Acoustic - John Adams (ID: 4Gcx0DkgBVticyPDpezwg5)
   Score: 0.15914731465723456
   Genre: acoustic
   Mood: sad
   Energy: 0.241


In [60]:
# Ranking Test 3: Multiple features, context 1
user = "joseph"

time = "morning"

query = {
    "genres": ["indie"],
    "mood": "",
    "energy": "medium"
}

results = rank_songs(query, top_k=5, user_id=user, time_of_day=time)

print_result_data(results)

1. Just You and I - Tom Walker (ID: 03x2rVJRFUrvwlfxoHd9Mo)
   Score: 0.1078452470985632
   Genre: indie-pop
   Mood: focus
   Energy: 0.696
2. Leave a Light On - Tom Walker (ID: 6lOWoTqVnAWXchddtTH31W)
   Score: 0.0948923376824682
   Genre: indie-pop
   Mood: focus
   Energy: 0.624
3. Cry Baby - The Neighbourhood (ID: 0EfsDEYaSjGYd66Pr881nq)
   Score: 0.0921216036976855
   Genre: alternative
   Mood: focus
   Energy: 0.656
4. Castle - Halsey (ID: 16w8ZGVSjI4TlTLV8VimBY)
   Score: 0.09152353134489388
   Genre: indie-pop
   Mood: focus
   Energy: 0.571
5. Now Or Never - Halsey (ID: 7i2DJ88J7jQ8K7zqFX2fW8)
   Score: 0.09144220418445215
   Genre: indie-pop
   Mood: focus
   Energy: 0.585


In [63]:
# Ranking Test 4: Multiple features, context 2
user = "joseph"

time = "night"

query = {
    "genres": ["indie"],
    "mood": "",
    "energy": "medium"
}

results = rank_songs(query, top_k=5, user_id=user, time_of_day=time)

print_result_data(results)

1. Just You and I - Tom Walker (ID: 03x2rVJRFUrvwlfxoHd9Mo)
   Score: 0.1078452470985632
   Genre: indie-pop
   Mood: focus
   Energy: 0.696
2. Alone - Kato (ID: 7kkT0Z4U36CLd8uhnQ9SMs)
   Score: 0.09959696194688897
   Genre: anime
   Mood: focus
   Energy: 0.617
3. Leave a Light On - Tom Walker (ID: 6lOWoTqVnAWXchddtTH31W)
   Score: 0.0948923376824682
   Genre: indie-pop
   Mood: focus
   Energy: 0.624
4. Castle - Halsey (ID: 16w8ZGVSjI4TlTLV8VimBY)
   Score: 0.09152353134489388
   Genre: indie-pop
   Mood: focus
   Energy: 0.571
5. Now Or Never - Halsey (ID: 7i2DJ88J7jQ8K7zqFX2fW8)
   Score: 0.09144220418445215
   Genre: indie-pop
   Mood: focus
   Energy: 0.585
